# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [1]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [3]:
import numpy as np
import itertools
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [4]:
def build_classifiers():
    all_models = [
        LinearRegression(),
        KNeighborsClassifier(),
        SVC(kernel='linear'),
        DecisionTreeClassifier(random_state=42),
        GaussianNB(),
        QuadraticDiscriminantAnalysis()
    ]
    
    best_accuracy = 0
    best_classifiers = None
    
    # Generate all possible combinations of 3 classifiers
    for combo in itertools.combinations(all_models, 3):
        classifiers = list(combo)
        
        # Fit the base classifiers
        for clf in classifiers:
            clf.fit(data_set, labels)
        
        # Get train predictions
        out = []
        for clf in classifiers:
            out.append(clf.predict(data_set))
        out = np.array(out).T.reshape((130, 3))
        
        # Train Meta-Classifier
        meta_clf = DecisionTreeClassifier(random_state=42)
        meta_clf.fit(out, labels.reshape((130,)))
        
        # Get test predictions
        t_out = []
        for clf in classifiers:
            t_out.append(clf.predict(test_data_set))
        t_out = np.array(t_out).T.reshape((len(t_out[0]), 3))
        
        # Check Accuracy
        acc = accuracy_score(test_labels, meta_clf.predict(t_out))
        
        if acc > best_accuracy:
            best_accuracy = acc
            best_classifiers = classifiers

    # Print the winning results!
    best_names = [type(clf).__name__ for clf in best_classifiers]
    print(f"Highest Stacking Accuracy: {best_accuracy}")
    print(f"Best 3 Classifiers: {best_names}\n")
    
    return best_classifiers

In [9]:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).T.reshape((130,3))
    
    # stacked classifier part:
    stacked_classifier = DecisionTreeClassifier(random_state=42)
    stacked_classifier.fit(output.reshape((130,3)), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).T.reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [10]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)
print(accuracy)

Highest Stacking Accuracy: 0.95
Best 3 Classifiers: ['LinearRegression', 'KNeighborsClassifier', 'SVC']

0.95


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [11]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [12]:
def set_new_weights(model):
    # I is 1 if the prediction is wrong, 0 if it is correct
    I = (model.predict(test_set) != test_labels).astype(int)
    return (1 + I) / np.sum(1 + I)

Train the classifier with the code below:

In [ ]:
from sklearn.base import clone

classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    current_classifier = clone(classifier)
    model = train_model(current_classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)

[0.0006502  0.0006502  0.00130039 0.00130039 0.0006502  0.0006502
 0.0006502  0.0006502  0.00130039 0.0006502  0.0006502  0.00130039
 0.0006502  0.0006502  0.0006502  0.0006502  0.00130039 0.0006502
 0.00130039 0.0006502  0.0006502  0.00130039 0.0006502  0.00130039
 0.00130039 0.00130039 0.00130039 0.00130039 0.0006502  0.00130039
 0.0006502  0.0006502  0.00130039 0.0006502  0.00130039 0.00130039
 0.00130039 0.00130039 0.0006502  0.00130039 0.0006502  0.00130039
 0.0006502  0.00130039 0.00130039 0.00130039 0.0006502  0.0006502
 0.0006502  0.00130039 0.00130039 0.0006502  0.0006502  0.0006502
 0.00130039 0.0006502  0.0006502  0.0006502  0.0006502  0.0006502
 0.00130039 0.00130039 0.00130039 0.00130039 0.0006502  0.00130039
 0.0006502  0.00130039 0.0006502  0.00130039 0.0006502  0.00130039
 0.00130039 0.0006502  0.0006502  0.00130039 0.00130039 0.00130039
 0.00130039 0.00130039 0.0006502  0.00130039 0.0006502  0.00130039
 0.00130039 0.0006502  0.00130039 0.0006502  0.0006502  0.00130039


Set the validation data set:

In [14]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [15]:
def get_prediction(x):
    # Ask all 10 models to predict the outcome
    all_predictions = np.array([clf.predict(x) for clf in classifiers])
    
    # Sum the votes (how many models voted for class 1)
    votes_for_1 = np.sum(all_predictions, axis=0)
    
    # If half or more voted for class 1, return 1. Otherwise return 0.
    return (votes_for_1 >= len(classifiers) / 2).astype(int)

Test it:

In [16]:
prediction = get_prediction(validate_x)[0]

print(prediction)

1
